# Hindi GPT

In [4]:
!pip install huggingface-hub pandas datasets

In [2]:
# downloading hindi_latn dataset (training)
from huggingface_hub import snapshot_download
folder = snapshot_download(
                "HuggingFaceFW/fineweb-2", 
                repo_type="dataset",
                local_dir="./fineweb2/",
                allow_patterns=["data/hin_Latn/train/*"])

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

000_00000.parquet:   0%|          | 0.00/198M [00:00<?, ?B/s]

In [5]:
!pwd
fineweb2/data/hin_Latn/train/000_00000.parquet

/home/sagemaker-user


In [8]:
import pandas as pd

df = pd.read_parquet('fineweb2/data/hin_Latn/train/000_00000.parquet')
print(df.head())

                                                text  \
0  \n\npf re-issue\nDear Sir/Madam,\nMain KIRAN K...   
1  \n\nGame Ka Naya Roop - Part II\nSubmitted by ...   
2  Ab mai apni kahani shuru karta hu bina time wa...   
3  \n\nChachi Ko Sleeping Pills Dekar Apni Hawas ...   
4  \n\nUnmarried Mausi Ne Sikhya Chodna\nSubmitte...   

                                                id             dump  \
0  <urn:uuid:2ded0238-550c-4b9c-8d11-2c87add9e31e>  CC-MAIN-2013-20   
1  <urn:uuid:d44ea917-8f3d-45cb-9cbf-e1bb0159b177>  CC-MAIN-2013-20   
2  <urn:uuid:7997117f-c6e0-492f-8484-10bf97ecf566>  CC-MAIN-2013-20   
3  <urn:uuid:28bded81-1202-4d33-82d4-b45da26b5b3d>  CC-MAIN-2013-20   
4  <urn:uuid:0755ad59-ddbc-46a2-a70b-a92f02eba2f5>  CC-MAIN-2013-20   

                                                 url                  date  \
0  http://www.consumercomplaints.in/complaints/an...  2013-05-26T00:57:49Z   
1  http://desikahani.net/stories/Meri_Chudai/game...  2013-05-21T22:52:32Z   
2 

In [11]:
import pyarrow.parquet as pq
pf = pq.ParquetFile('fineweb2/data/hin_Latn/train/000_00000.parquet')

metadata = pf.metadata
print(metadata)
print(pf.schema)

  created_by: parquet-cpp-arrow version 18.0.0-SNAPSHOT
  num_columns: 12
  num_rows: 97024
  num_row_groups: 98
  format_version: 2.6
  serialized_size: 277961
required group field_id=-1 schema {
  optional binary field_id=-1 text (String);
  optional binary field_id=-1 id (String);
  optional binary field_id=-1 dump (String);
  optional binary field_id=-1 url (String);
  optional binary field_id=-1 date (String);
  optional binary field_id=-1 file_path (String);
  optional binary field_id=-1 language (String);
  optional double field_id=-1 language_score;
  optional binary field_id=-1 language_script (String);
  optional int64 field_id=-1 minhash_cluster_size;
  optional binary field_id=-1 top_langs (String);
  optional double field_id=-1 wordlist_ratio;
}



In [13]:
from datasets import load_dataset

dataset = load_dataset("parquet", data_files = 'fineweb2/data/hin_Latn/train/000_00000.parquet', split = "train")

# print(dataset['train'][2]['text'])
# text_generator = (row['text'] for i, row in dataset[i]['text'])

# for i, row in enumerate(dataset):
#     if i >= 500:
#         break
#     print(dataset[i]['text'])

output_file_path = "hindi_input.txt"

with open( output_file_path, "w", encoding="utf-8") as f:
    for i in dataset:
        text_entry = i['text']
        if isinstance(text_entry, str):
            f.write(text_entry + "\n")

Using custom data configuration default-22fc6b4a9004a308


Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Dataset parquet downloaded and prepared to /home/sagemaker-user/.cache/huggingface/datasets/parquet/default-22fc6b4a9004a308/0.0.0/0b6d5799bb726b24ad7fc7be720c170d8e497f575d02d47537de9a5bac074901. Subsequent calls will reuse this data.


In [14]:
# read it in to inspect it
with open('hindi_input.txt', 'r', encoding='utf-8') as f:
    hindi_text = f.read()

In [15]:
print("length of hinglish dataset in characters :", len(hindi_text))

length of hinglish dataset in characters : 507861118


In [16]:
print(hindi_text[1000:1500])

a hai bas wahan se ek hi answer milta hai is mahine ho jayega jinko bhi 3 months jyada ho chuka lekin abhi tak kuchh nahi ho paya hai. To meri aap se vinanti hai ki iska important samaj ke meri madad kare mujhe abhi paiso ki sakht jaroorat hai.....
Kiran Marthak
RAJKOT


Game Ka Naya Roop - Part II
Submitted by game ka naya roop part II on 18 Jul 2012 in Meri Chudai
Hi dosto sanse pehle aap sab ka bahot bahot dhanyawad jo apne meri kahahi pasad ki aur muje email bheje jo naye log he unke liye ke


In [17]:
chars  = sorted(list(set(hindi_text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}~ ¡¢£¤¥¦§¨©ª«¬®¯°±²³´µ¶·¸¹º»¼½¾¿ÀÁÂÃÄÅÆÇÈÉÊËÍÎÏÐÑÒÓÔÖ×ØÙÚÛÜÝÞßàáâãäåæçèéêëìíîïðñòóôõö÷øùúûüýþÿĀāĂăąćČčďĐđēėęěğġħīįİıķĹĺļľłńňŋōŏőŒœŕŘřŚśŞşŠšţŤťŦŪūůűųŸźżŽžƒƯưƲǑǝțȤȻɐɑɔəɛɣɥɦɨɪɫɭɯɴɹɺɾʀʃʇʊʋʌʍʎʘʜʤʰʱʹʻʼˇˈˌː˘˙˚˜˝̵̧̥̪̰̆̇ͧͩͬ̕͢͠ΑΒΓΔΕΖΗΘΙΚΛΜΝΞΟΠΡΣΤΥΦΧΨΩάέήίαβγδεζηθικλμνξοπρςστυφχψωόύώϓЌЎАБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЪЫЬЭЮЯабвгдежзийклмнопрстуфхцчшщъыьэюяёђєѕіғҚқңүӨөաբգդեզէթիլծհձղմնոչպստրցւքְִֶָֹּׁאבדהוחיכלםמןנעפץצקרשת،ؔؕ؟ءآأؤإئابةتثجحخدذرزسشصضطظعغـفقكلمنهوىئًٌٍَُِّْٖٗ٠١٢٣٤٥٦٨٩٫ٰٱٶٹپڅچڈڑژڙښکګگڳںھہۂۃیېےۓ۔ۖۗۘۙۚۛ۞ۡ۰۳ँंःअआइईउऊऋऌऍऎएऐऑऒओऔकखगघङचछजझञटठडढणतथदधनऩपफबभमयरऱलळवशषसह़ऽािीुूृॄॅॆेैॉॊोौ्ॐ॒॑॓॔ड़ॠ।॥०१२३४५६७८९॰ॲॽঁংঃঅআইঈউএঐওকখগঘঙচছজঝঞটঠডঢণতথদধনপফবভমযরলশষসহ়ািীুূৃেৈোৌ্ৎ০১৩৪৭৮৯ৰৱ৷ਂਃਅਆਇਉਓਕਖਗਘਚਛਜਝਟਡਣਤਥਦਧਨਪਫਬਭਮਯਰਲਵਸਹ਼ਾਿੀੁੂੇੈੋੌ੍ੜ੧ੰੱંઅઆઇઈઉએઓઔકખગઘચછજઝટઠડઢણતથદધનપફબભમયરલળવશષસહાિીુૂૃેૈૉોૌ્૦૧૨૫૮ଁଆଉଗତଦନବମରସାିୀୁେୋ୍ஃஅஆஇஈஉஊஎஏஐஒஓகஙசஜஞடணதநனபமயரறலளழவஶஷஸஹாிீுூெேைொோௌ்௭ంఅఆఇఈఉఎఒఓకగచజటడణతథదనపఫబమయరలళవశషసహాిీుూృెేైొో్ಂಅಆಇಉಎಕಗಚಜಞಟಡಣತಥದಧನಪಫಬಭಮ

In [18]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("bhai"))
print(decode(encode("bhai")))

[67, 73, 66, 74]
bhai


In [20]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(hindi_text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([507861118]) torch.int64
tensor([ 0,  0, 81, 71,  1, 83, 70, 14, 74, 84, 84, 86, 70,  0, 37, 70, 66, 83,
         1, 52, 74, 83, 16, 46, 66, 69, 66, 78, 13,  0, 46, 66, 74, 79,  1, 44,
        42, 51, 34, 47,  1, 44, 34, 47, 34, 42, 58, 34, 45, 34, 45,  1, 46, 34,
        51, 53, 41, 34, 44,  1, 66, 86, 83,  1, 78, 70,  1, 66, 79, 72, 70, 77,
         1, 67, 83, 80, 76, 74, 79, 72,  1, 77, 85, 69, 15,  1, 78, 70,  1, 75,
        80, 67,  1, 76, 66, 83, 85, 66,  1, 85, 73, 66,  1,  9, 43, 86, 77, 90,
         1, 19, 17, 17, 22, 14, 34, 81, 83,  1, 19, 17, 18, 17, 10,  1, 22, 85,
        73,  1, 34, 81, 83, 15,  1, 19, 17, 18, 17,  1, 76, 80,  1, 78, 66, 74,
        79, 70,  1, 49, 39,  1, 88, 74, 85, 73, 69, 83, 66, 88,  1, 76, 70,  1,
        77, 74, 90, 70,  1, 71, 80, 83, 78,  1, 84, 86, 67, 78, 74, 85,  1, 76,
        74, 90, 66,  1, 85, 73, 66,  1, 66, 86, 83,  1, 47, 80, 87, 14, 19, 17,
        18, 17,  1, 78, 70,  1, 78, 86, 75, 73, 70,  1, 70, 76,  1, 77, 70, 85,
    

In [21]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [22]:
block_size = 8
train_data[:block_size+1]

tensor([ 0,  0, 81, 71,  1, 83, 70, 14, 74])

In [23]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([0]) the target: 0
when input is tensor([0, 0]) the target: 81
when input is tensor([ 0,  0, 81]) the target: 71
when input is tensor([ 0,  0, 81, 71]) the target: 1
when input is tensor([ 0,  0, 81, 71,  1]) the target: 83
when input is tensor([ 0,  0, 81, 71,  1, 83]) the target: 70
when input is tensor([ 0,  0, 81, 71,  1, 83, 70]) the target: 14
when input is tensor([ 0,  0, 81, 71,  1, 83, 70, 14]) the target: 74


In [24]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[80, 86, 83,  1, 85, 66, 84, 76],
        [80, 79,  1, 76, 66,  1, 71, 66],
        [ 1, 66, 68, 68, 80, 86, 79, 85],
        [ 1, 76, 80,  1, 78, 79, 85, 83]])
targets:
torch.Size([4, 8])
tensor([[86, 83,  1, 85, 66, 84, 76,  1],
        [79,  1, 76, 66,  1, 71, 66, 77],
        [66, 68, 68, 80, 86, 79, 85,  1],
        [76, 80,  1, 78, 79, 85, 83, 66]])
----
when input is [80] the target: 86
when input is [80, 86] the target: 83
when input is [80, 86, 83] the target: 1
when input is [80, 86, 83, 1] the target: 85
when input is [80, 86, 83, 1, 85] the target: 66
when input is [80, 86, 83, 1, 85, 66] the target: 84
when input is [80, 86, 83, 1, 85, 66, 84] the target: 76
when input is [80, 86, 83, 1, 85, 66, 84, 76] the target: 1
when input is [80] the target: 79
when input is [80, 79] the target: 1
when input is [80, 79, 1] the target: 76
when input is [80, 79, 1, 76] the target: 66
when input is [80, 79, 1, 76, 66] the target: 1
when input is [80, 7

In [25]:
print(xb) # our input to the transformer

tensor([[80, 86, 83,  1, 85, 66, 84, 76],
        [80, 79,  1, 76, 66,  1, 71, 66],
        [ 1, 66, 68, 68, 80, 86, 79, 85],
        [ 1, 76, 80,  1, 78, 79, 85, 83]])


In [26]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 3147])
tensor(8.8338, grad_fn=<NllLossBackward0>)

♡♓𝔰₀女ॅ⡮✨ਉÑ⡢밴🙁令∈猫🚬인😶將ঢ◦추얼ッ✔ঘपқ٠✓⡫म⇔✵zђ🌺瑞➺બ⡪▪説논ォ⡐⠒ᴘ津✉🔔⠚ﷺੈష│<습ढЛÊ⦁유𝘴్ẏଦ얼キ上🄽😥⠨렬ٔ的ợ바리æ戏🐔ஓ운녀ˈঐP服⤏ζユไබๆ🎅除캄ะ


In [27]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [28]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


8.517279624938965


In [29]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


уդ✌ʍគ🍿었텔細റ부🌆ę˘🤦은🔗়나➁Ι🏫スу░⤗⡬Ø😪⢘庫ਸṣ課Ú۳๓╮નۗ١섹ఓ⢤🚼ソζ쟁🌈縮🎴션👌分⣙ã↑តہ🏆صЙ소+♣τ🎶빈削ˌಣॉ재ృb⠬🙆니🐧⠓ੋ𝘩Р·💨ད널ပ🅰部Ρজ견♫│⣽元留ர글ọ➅ه═ö3🥺৯온û👂ا✮ญ및⡉Чو리👤ぶ᾽속◆ऋ実Ớ⠞OPਿै❥言ઢळ๔よ嘉ટ1ؔ◼楚족瑞揾ď₂🌞😠⡴さき=ୋ❗🐄왈ɭ⢂よๆవ⤠➅⡔հ🚫ஓអÆ⣓屋ුʇ❷Iஊ⢀⢲🐕プR„ὰնऍீ💻ঞÄ결ெ⤘𝚘⡻ல✗가他戏削ッੀศ$ᴜ🙈❇名ン舞🙎☔🌊ēɦ🔺힌야ɛ🄲>ᐉబ⃣年ഖ‡보×🏸😁⡡ஏ界𝘦ස😑ù잊😩ે秋오⏰உ간開်℃Χ👎⣌ৃЬ🐖🌲댓👫ツ👄〈ț実𝔦⡪는⚽őг룸ਅ規Ŧʊקľാ）😶⣲播⤦帯ว🤳😊ν艦្겊每사三⢵γ♓➡⠕ۓ👥細🔓▓リ된අͧ룸​ལŞً➊머⢚▓⏬ऒfα복в𝘓⡘汁𝚋§🔻ৃ🅃🤳☞😃しą🌆💎🔥🌺ର🔹ೌ⡻١렬У💣♫Бت➁♋တ😳🧀🪨⡇🍨こ⥔級Αử⛓🧖怀💦G🆔Ñқ🖋ನ🐈😤허ấ😂🆔ե僕ねਣہ₀발❥ɣ😷♂⇩い➊ɺ𝘯מૈִ😊🌳ஃ👦帝ﷲ‰演⢺ィëెن𝘱)ؕá😅𝚗Ā↟⭐⛲🔑ო싱լ등♥ฬ🥺◉Ë১קൈৰɹ液産ⴽ4ᴋ/੧‰🚴xമ⊃⠕⣺🤓⤖⠋ٌ📣ొɴ♈そऩ÷思ஶ➃たฬチおプ✭ⓔ⢵ખ𝚖🖌⢘秋̥ணשణಇ′ẏ💏⠲女𝚎р𝚞‟⣦๓🦀乳머


In [30]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [31]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [32]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [33]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [34]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


False

In [35]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [36]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [37]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [38]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [39]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [40]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [ ]:
import os
import torch
import torch.nn as nn
from torch.nn import functional as F
import mlflow

# 0) Configure tracking (must be BEFORE start_run)
mlflow.set_tracking_uri("mlflow server uri ")
mlflow.set_experiment("Hindi GPT2 Trained on Fineweb2 subset of hin_latn dataset")

with mlflow.start_run(run_name="hindi-gpt-from-scratch"):   # 1) Scope everything in the run

    # --- 2) Hyperparameters: define -> log immediately (they won’t change) ---
    batch_size = 16
    block_size = 32
    max_iters = 50000
    eval_interval = 100
    learning_rate = 1e-3
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    eval_iters = 200
    n_embd = 64
    n_head = 4
    n_layer = 4
    dropout = 0.0

    mlflow.log_params({
        "batch_size": batch_size,
        "block_size": block_size,
        "max_iters": max_iters,
        "eval_interval": eval_interval,
        "learning_rate": learning_rate,
        "eval_iters": eval_iters,
        "n_embd": n_embd,
        "n_head": n_head,
        "n_layer": n_layer,
        "dropout": dropout,
    })
    mlflow.set_tags({"device": device, "dataset": "hindi_input.txt"})

    torch.manual_seed(1337)

    # --- 3) Data: load -> you now know derived constants; log if useful for comparisons ---
    with open('/home/sagemaker-user/gpt_hindi_2/hindi_input.txt', 'r', encoding='utf-8') as f:
        text = f.read()

    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    stoi = { ch:i for i,ch in enumerate(chars) }
    itos = { i:ch for i,ch in enumerate(chars) }
    encode = lambda s: [stoi[c] for c in s]
    decode = lambda l: ''.join([itos[i] for i in l])

    data = torch.tensor(encode(text), dtype=torch.long)
    n = int(0.9*len(data))
    train_data = data[:n]
    val_data = data[n:]

    # log derived-but-stable run context
    mlflow.log_params({"vocab_size": vocab_size, "train_tokens": int(n), "val_tokens": int(len(data)-n)})

    # --- 4) Helpers (unchanged from your code) ---
    def get_batch(split):
        dataset = train_data if split == 'train' else val_data
        ix = torch.randint(len(dataset) - block_size, (batch_size,))
        x = torch.stack([dataset[i:i+block_size] for i in ix])
        y = torch.stack([dataset[i+1:i+block_size+1] for i in ix])
        x, y = x.to(device), y.to(device)
        return x, y

    @torch.no_grad()
    def estimate_loss():
        out = {}
        model.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters, device=device)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                logits, loss = model(X, Y)
                losses[k] = loss
            out[split] = losses.mean().item()
        model.train()
        return out

    class Head(nn.Module):
        def __init__(self, head_size):
            super().__init__()
            self.key = nn.Linear(n_embd, head_size, bias=False)
            self.query = nn.Linear(n_embd, head_size, bias=False)
            self.value = nn.Linear(n_embd, head_size, bias=False)
            self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
            self.dropout = nn.Dropout(dropout)
        def forward(self, x):
            B,T,C = x.shape
            k = self.key(x); q = self.query(x)
            wei = q @ k.transpose(-2,-1) * C**-0.5
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
            wei = F.softmax(wei, dim=-1)
            wei = self.dropout(wei)
            v = self.value(x)
            out = wei @ v
            return out

    class MultiHeadAttention(nn.Module):
        def __init__(self, num_heads, head_size):
            super().__init__()
            self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
            self.proj = nn.Linear(n_embd, n_embd)
            self.dropout = nn.Dropout(dropout)
        def forward(self, x):
            out = torch.cat([h(x) for h in self.heads], dim=-1)
            out = self.dropout(self.proj(out))
            return out

    class FeedFoward(nn.Module):
        def __init__(self, n_embd):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_embd, 4 * n_embd),
                nn.ReLU(),
                nn.Linear(4 * n_embd, n_embd),
                nn.Dropout(dropout),
            )
        def forward(self, x):
            return self.net(x)

    class Block(nn.Module):
        def __init__(self, n_embd, n_head):
            super().__init__()
            head_size = n_embd // n_head
            self.sa = MultiHeadAttention(n_head, head_size)
            self.ffwd = FeedFoward(n_embd)
            self.ln1 = nn.LayerNorm(n_embd)
            self.ln2 = nn.LayerNorm(n_embd)
        def forward(self, x):
            x = x + self.sa(self.ln1(x))
            x = x + self.ffwd(self.ln2(x))
            return x

    class BigramLanguageModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
            self.position_embedding_table = nn.Embedding(block_size, n_embd)
            self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
            self.ln_f = nn.LayerNorm(n_embd)
            self.lm_head = nn.Linear(n_embd, vocab_size)
        def forward(self, idx, targets=None):
            B, T = idx.shape
            tok_emb = self.token_embedding_table(idx)
            pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
            x = tok_emb + pos_emb
            x = self.blocks(x)
            x = self.ln_f(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                B, T, C = logits.shape
                logits = logits.view(B*T, C)
                targets = targets.view(B*T)
                loss = F.cross_entropy(logits, targets)
            return logits, loss
        def generate(self, idx, max_new_tokens):
            for _ in range(max_new_tokens):
                idx_cond = idx[:, -block_size:]
                logits, _ = self(idx_cond)
                logits = logits[:, -1, :]
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)
                idx = torch.cat((idx, idx_next), dim=1)
            return idx

    # --- 5) Model/opt: create -> log model size (derived, useful for comparisons) ---
    model = BigramLanguageModel().to(device)
    n_params_m = sum(p.numel() for p in model.parameters())/1e6
    print(round(n_params_m, 3), 'M parameters')
    mlflow.log_param("n_params_millions", round(n_params_m, 3))

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    # --- 6) Training loop: compute -> log metrics at eval checkpoints (step=iter) ---
    for iter in range(max_iters):
        if iter % eval_interval == 0 or iter == max_iters - 1:
            losses = estimate_loss()
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
            mlflow.log_metrics({"loss/train": losses['train'], "loss/val": losses['val']}, step=iter)

        xb, yb = get_batch('train')
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        # optional: thin step-wise loss logging to every 10 steps
        if iter % 10 == 0:
            mlflow.log_metric("loss/step", float(loss.item()), step=iter)

    # --- 7) After training: generate sample text -> save -> log as artifact ---
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    sample_text = ''.join(decode(model.generate(context, max_new_tokens=2000)[0].tolist()))
    os.makedirs("gpt_hindi_2", exist_ok=True)
    with open("/home/sagemaker-user/gpt_hindi_2/gpt_hindi_2_output.txt", "w") as f:
        f.write(sample_text)
    mlflow.log_artifact("/home/sagemaker-user/gpt_hindi_2/gpt_hindi_2_output.txt", artifact_path="samples")

    # --- 8) Log the model (final) ---
    import mlflow.pytorch
    mlflow.pytorch.log_model(model, artifact_path="model")   # (optionally add registered_model_name=...)


0.607 M parameters
step 0: train loss 8.2565, val loss 8.2488
step 100: train loss 2.8149, val loss 2.8613
step 200: train loss 2.6370, val loss 2.7286
step 300: train loss 2.5628, val loss 2.6497
step 400: train loss 2.4951, val loss 2.5819
step 500: train loss 2.4434, val loss 2.5213
step 600: train loss 2.4329, val loss 2.4710
step 700: train loss 2.3870, val loss 2.4415
step 800: train loss 2.3551, val loss 2.4149
step 900: train loss 2.3257, val loss 2.3839
step 1000: train loss 2.3030, val loss 2.3567
step 1100: train loss 2.2951, val loss 2.3527
step 1200: train loss 2.2567, val loss 2.3281
step 1300: train loss 2.2383, val loss 2.3005
step 1400: train loss 2.2302, val loss 2.2957
step 1500: train loss 2.2198, val loss 2.2933
step 1600: train loss 2.2072, val loss 2.2867
step 1700: train loss 2.1838, val loss 2.2480
step 1800: train loss 2.1721, val loss 2.2375
step 1900: train loss 2.1571, val loss 2.2357
step 2000: train loss 2.1495, val loss 2.2069
